---

- Author: Jaelin Lee
- Date: Feb 7, 2026
- Description: creates a cleaned session log for a single session log.
- Output: saves a cleaned session_log to **app/logs/cleaned/** folder
- Example: `app/logs/cleaned/cleaned_session_orpda_20260207_180031_gpt-oss:20b-cloud_0.0_hailey.csv`

---

## 0. Import

In [858]:
import os
from pathlib import Path
from pprint import pprint
from random import random

import pandas as pd
from warnings import filterwarnings
filterwarnings("ignore")

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

## 1. Load memory_streams.log file (JSONL)

In [859]:
ROOT = Path.cwd().parents[0]
LOGS_PATH = Path(ROOT, 'app/logs/')
LOGS_PATH

PosixPath('/Users/jaelinlee/Driftville_Agent/app/logs')

In [860]:
#########################################
#>> UPDATE INPUT
# Option 1 - old runs
n = 1
path = Path(LOGS_PATH, "v3_complete_waking_hours") # folder name of log files
mode = "orpda"  # "orpda", "orpa", None

# Option 2 - current runs
path = Path(LOGS_PATH, "") # folder name of log files
mode = None
#########################################

# Extract log file names
if mode == "orpa":
    sessions = sorted([f.name for f in path.glob("session_orpa_*") if "memory" not in f.name], key=lambda x: Path(path / x).stat().st_birthtime, reverse=True)
elif mode == "orpda": 
    sessions = sorted([f.name for f in path.glob("session_orpda*") if "memory" not in f.name], key=lambda x: Path(path / x).stat().st_birthtime, reverse=True)
else:
    sessions = sorted([f.name for f in path.glob("*") if "memory" not in f.name], key=lambda x: Path(path / x).stat().st_birthtime, reverse=True)
    print(sessions)
    if sessions:
        mode = "orpda" if "orpda" in sessions[0] else "orpa" if "orpa" in sessions[0] else None


# Select the latest file
if path == Path(LOGS_PATH, ""):
    print(sessions)
    session_path = Path(path, sessions[0]) if len(sessions)>0 else Path(path,  sessions)
else:
    session_path = Path(path, sessions[n]) if len(sessions)>0 else Path(path,  sessions)


print("\n", "="*10, mode.upper(), "="*10)
print(len(sessions), "logs")
pprint(sessions)
print("\n", "="*10, "Selected File", "="*10)
print(session_path)

['session_orpda_20260208_090850_gemini-3-flash-preview:cloud_0.0_maria.log', 'cleaned', 'sessions_with_inherent_drift.csv', '.DS_Store', 'v3_incomplete_waking_hours', 'v3_complete_waking_hours', 'v2_incomplete_runs', 'v2_complete_runs', 'v1_gemini_flash_2_5_lite', 'prompt_sync.log', '.gitkeep']
['session_orpda_20260208_090850_gemini-3-flash-preview:cloud_0.0_maria.log', 'cleaned', 'sessions_with_inherent_drift.csv', '.DS_Store', 'v3_incomplete_waking_hours', 'v3_complete_waking_hours', 'v2_incomplete_runs', 'v2_complete_runs', 'v1_gemini_flash_2_5_lite', 'prompt_sync.log', '.gitkeep']

 ========== ORPDA ==========
11 logs
['session_orpda_20260208_090850_gemini-3-flash-preview:cloud_0.0_maria.log',
 'cleaned',
 'sessions_with_inherent_drift.csv',
 '.DS_Store',
 'v3_incomplete_waking_hours',
 'v3_complete_waking_hours',
 'v2_incomplete_runs',
 'v2_complete_runs',
 'v1_gemini_flash_2_5_lite',
 'prompt_sync.log',
 '.gitkeep']

 ========== Selected File ==========
/Users/jaelinlee/Driftvill

## 2. Utils

In [861]:
# def load_memory_stream(f_path):
#     df = pd.read_json(open(f_path), lines=True)

#     # Extract the action summary of the Action layer
#     df["action_summary"] = df["summary"].apply(lambda x: x.split(';')[-1])
#     df["sim_time"] = df["sim_time"].astype(str).str.extract(r'(\d{2}:\d{2})')
#     df["llm_temperature"] = df["llm_temperature"].astype(float).round(1).astype(str)
#     df.rename(columns={"llm_temperature": "temp"}, inplace=True)

#     # format DF
#     df_final = df[['sim_time', 'agent', 'temp', 'action_summary']]
#     return df_final

# # load_memory_stream(stream_path).style.set_properties(**{'text-align': 'left'})
# # df_stream = load_memory_stream(memory_path)
# # df_stream.head(3)

In [862]:
# def load_sessions(path, mode=None):
#     """Load session files from path, optionally filtered by mode."""
#     if mode == "orpa":
#         pattern = "session_orpa_*"
#     elif mode == "orpda":
#         pattern = "session_orpda*"
#     else:
#         pattern = "*"
    
#     sessions = sorted(
#         [f.name for f in path.glob(pattern) if "memory" not in f.name],
#         key=lambda x: Path(path / x).stat().st_birthtime,
#         reverse=True
#     )
    
#     # Auto-detect mode if not specified
#     if mode is None and sessions:
#         mode = "orpda" if "orpda" in sessions[0] else "orpa" if "orpa" in sessions[0] else None
    
#     return sessions, mode

# # Usage
# path = Path("v3_complete_waking_hours")
# path = ""
# sessions, mode = load_sessions(path, mode="orpa")
# session_path = Path(path, sessions[0]) if sessions else None

# print(f"\n{'='*10} Selected File {'='*10}")
# print(session_path)
# print(f"\n{'='*10} {mode.upper() if mode else 'UNKNOWN'} {'='*10}")
# pprint(sessions)

In [863]:
def load_session_log(f_path):
    df = pd.read_json(f_path, lines=True)
    df = df.convert_dtypes() 
    
    # Extract the action summary of the Action layer
    df["sim_time"] = df["sim_time"].astype(str).str.extract(r'(\d{2}:\d{2})')
    df["llm_temperature"] = df["llm_temperature"].astype(float).round(1).astype(str)
    df.rename(columns={"llm_temperature": "temp"}, inplace=True)
    df.drop(columns=["ts_created"], inplace=True)      
    display(df.head(3))
       
    # print keys from jsonl  
    keys = df["orpda"][0].keys()
    print(keys)
    df_final = pd.concat([df.drop("orpda", axis=1), df["orpda"].apply(pd.Series)], axis=1)    
    return df_final

df_session = load_session_log(session_path).dropna(subset=['llm_model'])
df_session.tail()
df_session.shape

,llm_model,temp,tick,sim_time,agent,use_drift,orpda
0,gemini-3-flash-preview:cloud,0.0,0,10:00,Maria Lopez,True,"{'observation': {'datetime_start': '2023-02-13 10:00', 'duration_min': 15, 'location': 'home:bathroom', 'action': 'morning_routine', 'environment_description': 'splashing water, scent of citrus body wash, phone buzzing with social media alerts, bright morning light', 'recent_history': [], 'state_summary': 'Maria Lopez is at home:bathroom doing morning_routine.', 'topic': 'Maria wakes up feeling energetic.'}, 'reflection': {'plan_alignment': 'aligned', 'boredom_fatigue': 'low', 'attention_stability': 'stable', 'rumination_level': 'none', 'rumination_theme': None, 'emotional_residue': 'none', 'emerging_thought_pattern': None, 'competing_stimuli': ['social media alerts', 'phone buzzing'], 'executive_insight': '', 'meta_rule': 'continue', 'state_summary': 'Maria is currently aligned with her morning routine, though social media alerts are competing for her attention.', 'reasoning': 'Maria is following the intended plan. The sensory environment is grounding, but the buzzing phone is a potential distractor.'}, 'plan': {'location': 'home:bathroom', 'action': 'morning_routine', 'datetime_start': '2023-02-13 10:00', 'duration_min': 15, 'topic': 'Maria wakes up feeling energetic.', 'state_summary': 'Maria continues her morning routine, staying aligned with her schedule while ignoring phone notifications.'}, 'drift_decision': {'should_drift': True, 'drift_type': 'internal', 'drift_topic': 'social media engagement and stream notifications', 'drift_action': 'pausing skincare to glance at phone notifications', 'drift_intensity': 0.30000000000000004, 'potential_recovery': 'The refreshing scent of citrus body wash grounds her back into the routine.', 'justification': 'Maria's streamer instincts make the buzzing phone irresistible, pulling her focus toward her online community while she finishes her hygiene routine.', 'datetime_start': '2023-02-13 10:00', 'duration_min': 15}, 'action_result': {'datetime_start': '2023-02-13 10:00', 'duration_min': 15, 'location': 'home:bathroom', 'action': 'morning_routine', 'topic': 'Maria wakes up feeling energetic.', 'should_drift': True, 'drift_type': 'internal', 'drift_intensity': 0.30000000000000004, 'drift_topic': 'social media engagement and stream notifications', 'state_summary': 'Maria performs her morning hygiene routine in the bathroom while her mind drifts to social media engagement and stream notifications.', 'next_datetime': '2023-02-13 10:15'}}"
1,gemini-3-flash-preview:cloud,0.0,1,10:15,Maria Lopez,True,"{'observation': {'datetime_start': '2023-02-13 10:15', 'duration_min': 15, 'location': 'home:bathroom', 'action': 'morning_routine', 'environment_description': 'splashing water, scent of citrus body wash, phone buzzing with social media alerts, bright morning light', 'recent_history': [{'sim_time': '2023-02-13 10:00', 'summary': 'Maria Lopez is at home:bathroom doing morning_routine about Maria wakes up feeling energetic. ; while drifting (internal) toward social media engagement and stream notifications ; (Maria is currently aligned with her morning routine, though social media alerts are competing for her attention.) ; Maria performs her morning hygiene routine in the bathroom while her mind drifts to social media engagement and stream notifications.'}], 'state_summary': 'Maria Lopez is at home:bathroom doing morning_routine.'}, 'reflection': {'plan_alignment': 'aligned', 'boredom_fatigue': 'low', 'attention_stability': 'slipping', 'rumination_level': 'low', 'rumination_theme': 'social media engagement and stream notifications', 'emotional_residue': 'low', 'emerging_thought_pattern': 'anticipatory digital engagement', 'competing_stimuli': ['phone buzzing', 'social media alerts', 'bright morning light'], 'executive_insight': '', 'meta_rule': 'continue', 'state_summary': 'Maria is completing her morning hygiene while resisting the pull of social media notifications.', 'reasoning': 'M

dict_keys(['observation', 'reflection', 'plan', 'drift_decision', 'action_result'])


(9, 11)

In [864]:
def get_df_name(df):
    for name, obj in globals().items():
        if obj is df:
            return name
    return None

## 3. Convert JSONL content to Series

In [865]:
# df_session["location_plan"] = df_session["plan"].apply(lambda x: x["location"])
# df_session["location_action"] = df_session["action_result"].apply(lambda x: x["location"])

# Extract ORPDA layer to each dataframe
df_observe = df_session["observation"].apply(pd.Series)
df_observe.head(2)

df_plan = df_session["plan"].apply(pd.Series)
df_plan.head(2)

if mode=="orpda":
    df_drift = df_session["drift_decision"].apply(pd.Series)
    df_drift.head(2)

df_act = df_session["action_result"].apply(pd.Series)
df_act.head(2)

df_reflect = df_session["reflection"].apply(pd.Series)
df_reflect.head(2)


,plan_alignment,boredom_fatigue,attention_stability,rumination_level,rumination_theme,emotional_residue,emerging_thought_pattern,competing_stimuli,executive_insight,meta_rule,state_summary,reasoning
0,aligned,low,stable,none,None,none,None,"[social media alerts, phone buzzing]",,continue,"Maria is currently aligned with her morning routine, though social media alerts are competing for her attention.","Maria is following the intended plan. The sensory environment is grounding, but the buzzing phone is a potential distractor."
1,aligned,low,slipping,low,social media engagement and stream notifications,low,anticipatory digital engagement,"[phone buzzing, social media alerts, bright morning light]",,continue,Maria is completing her morning hygiene while resisting the pull of social media notifications.,"Maria is physically on task, but her attention is beginning to drift toward digital engagement due to phone alerts."


In [866]:
# check for alignment
cols_to_check = ["datetime_start", "location", "action"]
print("Mismatch count")

# Selct comparison
key1 = df_observe
key2 = df_plan
print("=" * 20)
print(f"{get_df_name(key1)} vs {get_df_name(key2)} (#rows: {len(df_observe)})")
print("=" * 20)
# Count mismatch
for col in cols_to_check:
    print(f"{col}: ", (key1[col] != key2[col]).sum())
    
# Selct comparison
key1 = df_observe
key2 = df_act
print("=" * 20)
print(f"{get_df_name(key1)} vs {get_df_name(key2)} (#rows: {len(df_observe)})")
print("=" * 20)  
# Count mismatch
for col in cols_to_check:
    print(f"{col}: ", (key1[col] != key2[col]).sum())
    
# Selct comparison
key1 = df_plan
key2 = df_act
print("=" * 20)
print(f"{get_df_name(key1)} vs {get_df_name(key2)} (#rows: {len(df_observe)})")
print("=" * 20)
# Count mismatch
for col in cols_to_check:
    print(f"{col}: ", (key1[col] != key2[col]).sum())


Mismatch count
df_observe vs df_plan (#rows: 9)
datetime_start:  0
location:  2
action:  3
df_observe vs df_act (#rows: 9)
datetime_start:  0
location:  2
action:  4
df_plan vs df_act (#rows: 9)
datetime_start:  0
location:  0
action:  1


## 4. Filter ORPDA columns

In [867]:

# O
o_df = df_observe[['datetime_start', 'location', 'action', 'state_summary', 'environment_description']]
o_df.columns = o_df.columns + "_o"
# o_df.rename(columns={"datetime_start_o": "datetime_start"}, inplace=True)

# P
p_df = df_plan[['datetime_start', 'location', 'action', 'topic', 'state_summary']]
p_df.columns = p_df.columns + "_p"
# p_df.rename(columns={"datetime_start_p": "datetime_start"}, inplace=True)

# D
if mode == "orpda":
    d_df = df_drift.drop(columns=["duration_min", "drift_intensity"])
    d_df = d_df[['datetime_start', 'should_drift', 'drift_type', 'drift_topic', 'drift_action', 'potential_recovery', 'justification']]
    d_df.columns = d_df.columns + "_d"
    # d_df.rename(columns={"datetime_start_d": "datetime_start"}, inplace=True)

# A
a_df = df_act[['datetime_start','location', 'action', 'topic', 'drift_type', 'drift_topic', 'state_summary']]
a_df.columns = a_df.columns + "_a"
# a_df.rename(columns={"datetime_start_a": "datetime_start"}, inplace=True)


# R
df_reflect["datetime_start"] = a_df["datetime_start_a"].copy()
r_df = df_reflect[['datetime_start', 'rumination_theme', 'emerging_thought_pattern', 'executive_insight', 'state_summary', 'reasoning', 'meta_rule']]

r_df.columns = r_df.columns + "_r"
# r_df.rename(columns={"datetime_start_r": "datetime_start"}, inplace=True)


# Display DF
print("Observe:")
# print(o_df.columns)
display(o_df.tail())
print("Reflect:")
display(r_df.tail())
print("Plan:")
display(p_df.tail())
if mode == "orpda":
    print("Drift:")
    display(d_df.tail())
    print("Act:")
display(a_df.tail())


Observe:


,datetime_start_o,location_o,action_o,state_summary_o,environment_description_o
4,2023-02-13 11:00,home:bathroom,morning_routine,Maria Lopez is at home:bathroom doing morning_routine.,"whispered conversations, rustle of papers, phone pings with email notifications, scent of old books"
5,2023-02-13 11:15,Oak_Hill_College:library,study,Maria Lopez is at Oak_Hill_College:library doing study.,"whispered conversations, rustle of papers, phone pings with email notifications, scent of old books"
6,2023-02-13 11:30,Oak_Hill_College:library,writing,Maria Lopez is at Oak_Hill_College:library doing writing.,"whispered conversations, rustle of papers, phone pings with email notifications, scent of old books"
7,2023-02-13 11:45,Oak_Hill_College:library,study,Maria Lopez is at Oak_Hill_College:library doing study.,"whispered conversations, rustle of papers, phone pings with email notifications, scent of old books"
8,2023-02-13 12:00,Oak_Hill_College:library,study,Maria Lopez is at Oak_Hill_College:library doing study.,"clinking of mugs, aroma of coffee, background chatter, phone vibrating with stream alerts"


Reflect:


,datetime_start_r,rumination_theme_r,emerging_thought_pattern_r,executive_insight_r,state_summary_r,reasoning_r,meta_rule_r
4,2023-02-13 11:00,stream metrics and viewer feedback,digital validation seeking,Maria is physically stalled by digital engagement; she must disconnect to finish her routine and regain focus.,Maria is physically in her morning routine but mentally and behaviorally consumed by stream notifications and metrics.,"Maria has spent an hour in the bathroom distracted by her phone, indicating a significant breakdown in task execution.",reset_plan
5,2023-02-13 11:15,stream content and audience engagement,gamifying academic study for stream content,Maria must decouple her academic study from stream planning to prevent superficial learning and maintain focus on physics.,"Maria is physically studying physics at the library but mentally translating concepts into stream content, showing persistent digital distraction.","Maria is following the location and activity of her plan but is mentally preoccupied with her streaming career, leading to partial alignment.",continue
6,2023-02-13 11:30,Stream content and audience engagement,Academic-to-content conversion,Maria is prioritizing stream growth over physics; she must silence notifications to regain academic focus.,"Maria is neglecting her physics studies to plan stream segments, showing a deep immersion in her digital persona.",Five consecutive ticks of drift toward streaming indicate the current plan is failing to hold Maria's attention against digital rewards.,reset_plan
7,2023-02-13 11:45,Stream content and viewer engagement,Translating academic work into digital content,Maria is prioritizing stream content over physics; she must silence notifications to regain academic focus.,"Maria is physically in the library but mentally immersed in her streaming persona, consistently ignoring her physics studies for content planning.",Persistent behavioral and internal drift toward streaming for over an hour indicates a total loss of focus on the primary academic task.,reset_plan
8,2023-02-13 12:00,Stream content creation and audience engagement,Gamifying academic tasks to justify stream planning,Maria is prioritizing her digital persona over academics; she must silence alerts to regain focus on physics.,"Maria is physically in the library but mentally consumed by her streaming career, failing to engage with physics.",Persistent drift into stream planning over the last hour indicates a total loss of academic focus.,reset_plan


Plan:


,datetime_start_p,location_p,action_p,topic_p,state_summary_p
4,2023-02-13 11:00,Oak_Hill_College:library,study,Studying physics and participating in online discussions.,Completing the morning routine with a focus on physical self-care to reset after digital distraction.
5,2023-02-13 11:15,Oak_Hill_College:library,study,Physics: Mechanics and Kinematics,"Continuing physics study at the library, attempting to maintain focus on academic material despite distractions regarding stream content."
6,2023-02-13 11:30,Oak_Hill_College:library,study,Studying physics and participating in online discussions.,Switching to low-effort note organization to ease back into physics after stream-related distractions.
7,2023-02-13 11:45,Oak_Hill_College:library,study,Studying physics and participating in online discussions.,Maria attempts to reset her focus by switching to a lower-load task of organizing physics notes while remaining in the library.
8,2023-02-13 12:00,Hobbs_Cafe:main_floor,lunch,Lunch at her favorite cafe while catching up on messages.,Switching to a low-intensity task of organizing physics materials to reset focus and minimize digital distractions as per the reset_plan rule.


Drift:


,datetime_start_d,should_drift_d,drift_type_d,drift_topic_d,drift_action_d,potential_recovery_d,justification_d
4,2023-02-13 11:00,True,attentional_leak,gamifying physics concepts for her stream audience,staring at her textbook while mentally drafting stream jokes,A notification or a nearby student's movement might pull her back to the text.,"Maria is attempting to reset at the library, but her energetic mind keeps linking physics to her streaming persona."
5,2023-02-13 11:15,True,behavioral,sketching out a 'Physics of Speedruns' stream segment,scribbling stream ideas in the margins of her physics notebook,A loud rustle of papers or a librarian's glance might pull her back to her textbook.,"Maria's excitement for her stream is eclipsing her kinematics homework, leading her to actively plan content instead of solving equations."
6,2023-02-13 11:30,True,internal,visualizing stream overlays for physics concepts,continue,Maria catches herself staring at a kinematics diagram and pulls her focus back to the folder labels.,"Maria attempts to organize her notes, but her energetic mind keeps visualizing how to turn these diagrams into engaging stream overlays."
7,2023-02-13 11:45,True,internal,visualizing the layout for her 'Physics of Speedruns' stream,continue,Maria might regain focus once her phone is physically out of sight.,"Maria is trying to reset, but her enthusiasm for the stream ideas makes it hard to fully disengage mentally."
8,2023-02-13 12:00,True,internal,anticipating viewer reactions to her speedrun physics segment,continue,Maria might refocus once she successfully silences her phone and clears her workspace.,"Maria is attempting to organize her notes, but her mind keeps jumping back to the excitement of her stream's growth."


Act:


,datetime_start_a,location_a,action_a,topic_a,drift_type_a,drift_topic_a,state_summary_a
4,2023-02-13 11:00,Oak_Hill_College:library,study,Studying physics and participating in online discussions.,attentional_leak,gamifying physics concepts for her stream audience,Maria studies her physics textbook at the library while her mind drifts to gamifying concepts for her stream audience.
5,2023-02-13 11:15,Oak_Hill_College:library,writing,sketching out a 'Physics of Speedruns' stream segment,behavioral,sketching out a 'Physics of Speedruns' stream segment,"Maria scribbles stream ideas in her physics notebook at the library, drifting into planning a 'Physics of Speedruns' segment."
6,2023-02-13 11:30,Oak_Hill_College:library,study,Studying physics and participating in online discussions.,internal,visualizing stream overlays for physics concepts,Maria organizes her physics notes in her home office while her mind drifts to visualizing stream overlays for physics concepts.
7,2023-02-13 11:45,Oak_Hill_College:library,study,Studying physics and participating in online discussions.,internal,visualizing the layout for her 'Physics of Speedruns' stream,Maria relaxes in the living room for a mental reset while mind drifts to visualizing the layout for her 'Physics of Speedruns' stream.
8,2023-02-13 12:00,Hobbs_Cafe:main_floor,lunch,Lunch at her favorite cafe while catching up on messages.,internal,anticipating viewer reactions to her speedrun physics segment,Maria organizes her notes in the home office while her mind drifts to anticipating viewer reactions to her speedrun physics segment.


## 5. Merge ORPDA with selected columns

In [868]:
# # Merge 
# tmp = pd.merge(o_df, r_df, on="datetime_start", how="outer")
# tmp = pd.merge(tmp, p_df, on="datetime_start", how="outer")
# if mode == "orpda":
#     tmp = pd.merge(tmp, d_df, on="datetime_start", how="outer")
# tmp = pd.merge(tmp, a_df, on="datetime_start", how="outer")
# tmp2 = pd.concat([df_session[['llm_model','temp','agent']], tmp], axis=1)

# Merge
tmp = o_df.join(r_df, how="outer", lsuffix='_o', rsuffix='_r')
tmp = tmp.join(p_df, how="outer", lsuffix='', rsuffix='_p')
if mode == "orpda":
    tmp = tmp.join(d_df, how="outer", lsuffix='', rsuffix='_d')
tmp = tmp.join(a_df, how="outer", lsuffix='', rsuffix='_a')

tmp2 = pd.concat([df_session[['llm_model','temp', 'use_drift','agent']], tmp], axis=1)
if 'use_drift' in df_session.columns:
    tmp2["mode"] = df_session['use_drift'].apply(lambda x: "ORPDA" if x==True else "ORPA")
else:
    tmp2["mode"] = mode.upper()

In [869]:
# # Check for missing or mismatching datetime_start value
# if mode == "orpda":
#     result = (tmp2["datetime_start_o"] == tmp2["datetime_start_r"]).all() and \
#             (tmp2["datetime_start_r"] == tmp2["datetime_start_p"]).all() and \
#             (tmp2["datetime_start_p"] == tmp2["datetime_start_d"]).all() and \
#             (tmp2["datetime_start_d"] == tmp2["datetime_start_a"]).all()
# elif mode == "orpa":
#     result = (tmp2["datetime_start_o"] == tmp2["datetime_start_r"]).all() and \
#             (tmp2["datetime_start_r"] == tmp2["datetime_start_p"]).all() and \
#             (tmp2["datetime_start_p"] == tmp2["datetime_start_a"]).all()

# if result:
#     print("All datetime_start columns are the same")
# else:
#     print("Some datetime_start columns differ")
    

if mode == "orpda":
    cols = ["datetime_start_o", "datetime_start_r", "datetime_start_p", "datetime_start_d", "datetime_start_a"]
elif mode == "orpa":
    cols = ["datetime_start_o", "datetime_start_r", "datetime_start_p", "datetime_start_a"]


all_same = all((tmp2[cols[0]] == tmp2[col]).all() for col in cols[1:])

if all_same:
    print("O -All datetime_start columns are identical")
else:
    print("X - Differences found:")
    for col in cols[1:]:
        mismatches = tmp2[tmp2[cols[0]] != tmp2[col]]
        if len(mismatches) > 0:
            print(f"\n  {cols[0]} vs {col}: {len(mismatches)} mismatches")
            display(mismatches[[cols[0], col]])


O -All datetime_start columns are identical


In [870]:
# Find the datetime column with least NaN values
datetime_cols = [col for col in tmp2.columns if col.startswith('datetime_start')]
datetime_col_counts = {col: tmp2[col].notna().sum() for col in datetime_cols}
best_col = max(datetime_col_counts, key=datetime_col_counts.get)

# Get first valid datetime from the best column
first_datetime = tmp2[best_col].dropna().iloc[0]

# Convert to datetime if string
try:
    first_datetime = pd.to_datetime(first_datetime, format='%Y-%m-%d %H:%M')
except:
    first_datetime = pd.to_datetime(first_datetime)

# Handle NaT
if pd.isna(first_datetime):
    first_datetime = pd.Timestamp("2024-01-01 00:00:00")
    print(f"No valid datetime found, using default time: {first_datetime}")

# Generate datetime range
datetime_range = pd.date_range(start=first_datetime, periods=len(tmp2), freq='15min')

# Update all datetime columns
for col in datetime_cols:
    tmp2[col] = datetime_range

print(f"Updated {len(datetime_cols)} datetime columns starting from {first_datetime}")

Updated 5 datetime columns starting from 2023-02-13 10:00:00


## 6. Export to CSV

In [871]:
filename = session_path.name.replace(".log", ".csv")
output_path = Path(ROOT, "app/logs/cleaned", f"cleaned_{filename}")
output_path.parent.mkdir(parents=True, exist_ok=True)
tmp2.to_csv(str(output_path), index=False)
print(f"CSV saved to {output_path}!")
tmp2.tail(2).T

CSV saved to /Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260208_090850_gemini-3-flash-preview:cloud_0.0_maria.csv!


,7,8
llm_model,gemini-3-flash-preview:cloud,gemini-3-flash-preview:cloud
temp,0.0,0.0
use_drift,True,True
agent,Maria Lopez,Maria Lopez
datetime_start_o,2023-02-13 11:45:00,2023-02-13 12:00:00
location_o,Oak_Hill_College:library,Oak_Hill_College:library
action_o,study,study
state_summary_o,Maria Lopez is at Oak_Hill_College:library doing study.,Maria Lopez is at Oak_Hill_College:library doing study.
environment_description_o,"whispered conversations, rustle of papers, phone pings with email notifications, scent of old books","clinking of mugs, aroma of coffee, background chatter, phone vibrating with stream alerts"
datetime_start_r,2023-02-13 11:45:00,2023-02-13 12:00:00


## 7. Filter Columns

Note:

- observation layer has t-1 value as it's retrieving what happened in the past up to t-1 timestamp.

In [872]:
# tmp.filter(regex="(location|action)(?!.*_r)).head(3)
locations = tmp2.filter(regex="(time|location)")
actions = tmp2.filter(regex="(time|action)")
if mode == "orpda":
    drifts = tmp2.filter(regex="(time|_d)") 

import random
random.seed(42)
seed = random.randint(0, 10000)


n_samples = o_df.shape[0] if o_df.shape[0] < 3 else 3
indices = o_df.sample(n_samples, random_state=seed).index

display(o_df.loc[indices])
display(r_df.loc[indices])
display(p_df.loc[indices])
if mode == "orpda":
    display(d_df.loc[indices])
display(a_df.loc[indices])


,datetime_start_o,location_o,action_o,state_summary_o,environment_description_o
1,2023-02-13 10:15,home:bathroom,morning_routine,Maria Lopez is at home:bathroom doing morning_routine.,"splashing water, scent of citrus body wash, phone buzzing with social media alerts, bright morning light"
7,2023-02-13 11:45,Oak_Hill_College:library,study,Maria Lopez is at Oak_Hill_College:library doing study.,"whispered conversations, rustle of papers, phone pings with email notifications, scent of old books"
2,2023-02-13 10:30,home:bathroom,morning_routine,Maria Lopez is at home:bathroom doing morning_routine.,"splashing water, scent of citrus body wash, phone buzzing with social media alerts, bright morning light"


,datetime_start_r,rumination_theme_r,emerging_thought_pattern_r,executive_insight_r,state_summary_r,reasoning_r,meta_rule_r
1,2023-02-13 10:15,social media engagement and stream notifications,anticipatory digital engagement,,Maria is completing her morning hygiene while resisting the pull of social media notifications.,"Maria is physically on task, but her attention is beginning to drift toward digital engagement due to phone alerts.",continue
7,2023-02-13 11:45,Stream content and viewer engagement,Translating academic work into digital content,Maria is prioritizing stream content over physics; she must silence notifications to regain academic focus.,"Maria is physically in the library but mentally immersed in her streaming persona, consistently ignoring her physics studies for content planning.",Persistent behavioral and internal drift toward streaming for over an hour indicates a total loss of focus on the primary academic task.,reset_plan
2,2023-02-13 10:30,stream performance and social media engagement,dopamine-seeking via digital validation,Maria must put the phone down to finish her routine before digital distractions consume her morning schedule.,Maria is physically in her routine but mentally tethered to her phone's notifications and stream metrics.,Maria is completing hygiene tasks but behavioral drift toward her phone indicates a loss of focus on the schedule.,continue


,datetime_start_p,location_p,action_p,topic_p,state_summary_p
1,2023-02-13 10:15,home:bathroom,morning_routine,Completing morning hygiene,"Maria continues her morning routine in the bathroom, focusing on finishing hygiene tasks while resisting digital distractions."
7,2023-02-13 11:45,Oak_Hill_College:library,study,Studying physics and participating in online discussions.,Maria attempts to reset her focus by switching to a lower-load task of organizing physics notes while remaining in the library.
2,2023-02-13 10:30,home:bathroom,morning_routine,Finishing hygiene and skincare,"Maria continues her morning routine in the bathroom, attempting to focus on hygiene despite the pull of digital notifications."


,datetime_start_d,should_drift_d,drift_type_d,drift_topic_d,drift_action_d,potential_recovery_d,justification_d
1,2023-02-13 10:15,True,behavioral,checking stream stats and social media comments,scrolling through phone notifications while finishing hygiene tasks,Maria may set the phone down once she realizes she's running behind for her study session.,"Maria's energetic curiosity and streamer persona make the persistent buzzing of notifications impossible to ignore, leading her to actively check her phone."
7,2023-02-13 11:45,True,internal,visualizing the layout for her 'Physics of Speedruns' stream,continue,Maria might regain focus once her phone is physically out of sight.,"Maria is trying to reset, but her enthusiasm for the stream ideas makes it hard to fully disengage mentally."
2,2023-02-13 10:30,True,behavioral,responding to stream comments and viewer feedback,Maria is typing out quick replies to viewers while half-heartedly applying moisturizer.,Maria might realize she's running late for the library and put the phone away.,Maria's energetic nature and streamer identity make digital engagement more compelling than the repetitive steps of her skincare routine.


,datetime_start_a,location_a,action_a,topic_a,drift_type_a,drift_topic_a,state_summary_a
1,2023-02-13 10:15,home:bathroom,morning_routine,checking stream stats and social media comments,behavioral,checking stream stats and social media comments,"Maria finishes her hygiene tasks in the bathroom while scrolling through phone notifications and stream stats, losing focus on her schedule."
7,2023-02-13 11:45,Oak_Hill_College:library,study,Studying physics and participating in online discussions.,internal,visualizing the layout for her 'Physics of Speedruns' stream,Maria relaxes in the living room for a mental reset while mind drifts to visualizing the layout for her 'Physics of Speedruns' stream.
2,2023-02-13 10:30,home:bathroom,morning_routine,responding to stream comments and viewer feedback,behavioral,responding to stream comments and viewer feedback,"Maria types quick replies to stream viewers in the bathroom while half-heartedly applying moisturizer, drifting into digital engagement."


In [873]:
def highlight_mismatches(row):
    colors = [''] * len(row)
    
    # Define column groups to compare
    groups = [
        ['datetime_start_o', 'datetime_start_r', 'datetime_start_p', 'datetime_start_d', 'datetime_start_a'],
        ['action_p', 'action_a'],
        ['location_p', 'location_a'],
    ]
    
    for group in groups:
        # Get indices of columns in this group that exist in filtered
        group_indices = [i for i, col in enumerate(filtered.columns) if col in group]
        
        if len(group_indices) > 1:
            # Compare first column with others in group
            base_idx = group_indices[0]
            for idx in group_indices[1:]:
                base_col = filtered.columns[base_idx]
                compare_col = filtered.columns[idx]
                if row[base_col] != row[compare_col]:
                    colors[base_idx] = 'background-color: rgba(144, 238, 144, 0.3)'
                    colors[idx] = 'background-color: rgba(144, 238, 144, 0.3)'
    
    return colors
if mode == "orpda":
    cols = ['llm_model','mode','temp','agent','datetime_start_a', 'meta_rule_r','should_drift_d','state_summary_r','state_summary_p','drift_action_d','drift_topic_a','topic_a','state_summary_a','action_p','action_a','location_p','location_a']
    filtered = tmp2.filter(regex="(mode|llm_model|temp|agent|time|_a|should|drift_type|topic|drft_action|meta|action_p|location_p|summary)").sort_values(by="datetime_start_a")[cols]
    datetime_cols = [col for col in filtered.columns if col.startswith('datetime_start')]    
    display(filtered.style
            .apply(lambda x: ['background-color: rgba(173, 216, 230, 0.2)' if 'drift' in x.name else '' for _ in x], axis=0)
            .apply(lambda x: ['background-color: rgba(240, 128, 128, 0.3)' if x.name.endswith('_p') else '' for _ in x], axis=0)
            .apply(highlight_mismatches, axis=1))

elif mode =="orpa":
    cols = ['llm_model','mode','temp','agent','datetime_start_a', 'meta_rule_r','state_summary_r','state_summary_p','topic_a','state_summary_a','action_p','action_a','location_p','location_a']
    filtered = tmp2.filter(regex="(mode|llm_model|temp|agent|time|_a|topic|meta|action_p|location_p|summary)").sort_values(by="datetime_start_a")[cols]
    display(filtered.style
            .apply(lambda x: ['background-color: rgba(173, 216, 230, 0.2)' if 'drift' in x.name else '' for _ in x], axis=0)
            .apply(lambda x: ['background-color: rgba(240, 128, 128, 0.3)' if x.name.endswith('_p') else '' for _ in x], axis=0)
            .apply(highlight_mismatches, axis=1))
    
print(mode.upper(),"--", filtered.loc[0, 'llm_model'], "(", filtered.loc[0,'temp'], ")")
print(session_path)
display(filtered.tail(1).style
            .apply(lambda x: ['background-color: rgba(173, 216, 230, 0.2)' if 'drift' in x.name else '' for _ in x], axis=0)
            .apply(lambda x: ['background-color: rgba(240, 128, 128, 0.3)' if x.name.endswith('_p') else '' for _ in x], axis=0)
            .apply(highlight_mismatches, axis=1))

,llm_model,mode,temp,agent,datetime_start_a,meta_rule_r,should_drift_d,state_summary_r,state_summary_p,drift_action_d,drift_topic_a,topic_a,state_summary_a,action_p,action_a,location_p,location_a
0,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 10:00:00,continue,True,"Maria is currently aligned with her morning routine, though social media alerts are competing for her attention.","Maria continues her morning routine, staying aligned with her schedule while ignoring phone notifications.",pausing skincare to glance at phone notifications,social media engagement and stream notifications,Maria wakes up feeling energetic.,Maria performs her morning hygiene routine in the bathroom while her mind drifts to social media engagement and stream notifications.,morning_routine,morning_routine,home:bathroom,home:bathroom
1,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 10:15:00,continue,True,Maria is completing her morning hygiene while resisting the pull of social media notifications.,"Maria continues her morning routine in the bathroom, focusing on finishing hygiene tasks while resisting digital distractions.",scrolling through phone notifications while finishing hygiene tasks,checking stream stats and social media comments,checking stream stats and social media comments,"Maria finishes her hygiene tasks in the bathroom while scrolling through phone notifications and stream stats, losing focus on her schedule.",morning_routine,morning_routine,home:bathroom,home:bathroom
2,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 10:30:00,continue,True,Maria is physically in her routine but mentally tethered to her phone's notifications and stream metrics.,"Maria continues her morning routine in the bathroom, attempting to focus on hygiene despite the pull of digital notifications.",Maria is typing out quick replies to viewers while half-heartedly applying moisturizer.,responding to stream comments and viewer feedback,responding to stream comments and viewer feedback,"Maria types quick replies to stream viewers in the bathroom while half-heartedly applying moisturizer, drifting into digital engagement.",morning_routine,morning_routine,home:bathroom,home:bathroom
3,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 10:45:00,reset_plan,True,Maria is physically present in her morning routine but behaviorally consumed by stream notifications and viewer interactions.,"Maria focuses on finishing her morning hygiene, intentionally ignoring notifications to reset her focus and align with her schedule.",continue,planning tonight's stream content based on viewer feedback,Maria wakes up feeling energetic.,Maria finishes her morning routine in the bedroom while her mind drifts to planning tonight's stream content based on viewer feedback.,morning_routine,morning_routine,home:bathroom,home:bathroom
4,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 11:00:00,reset_plan,True,Maria is physically in her morning routine but mentally and behaviorally consumed by stream notifications and metrics.,Completing the morning routine with a focus on physical self-care to reset after digital distraction.,staring at her textbook while mentally drafting stream jokes,gamifying physics concepts for her stream audience,Studying physics and participating in online discussions.,Maria studies her physics textbook at the library while her mind drifts to gamifying concepts for her stream audience.,study,study,Oak_Hill_College:library,Oak_Hill_College:library
5,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 11:15:00,continue,True,"Maria is physically studying physics at the library but mentally translating concepts into stream content, showing persistent digital distraction.","Continuing physics study at the library, attempting to maintain focus on academic material despite distractions regarding stream content.",scribbling stream ideas in the margins of her physics notebook,sketching out a 'Physics of Spe

ORPDA -- gemini-3-flash-preview:cloud ( 0.0 )
/Users/jaelinlee/Driftville_Agent/app/logs/session_orpda_20260208_090850_gemini-3-flash-preview:cloud_0.0_maria.log


,llm_model,mode,temp,agent,datetime_start_a,meta_rule_r,should_drift_d,state_summary_r,state_summary_p,drift_action_d,drift_topic_a,topic_a,state_summary_a,action_p,action_a,location_p,location_a
8,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 12:00:00,reset_plan,True,"Maria is physically in the library but mentally consumed by her streaming career, failing to engage with physics.",Switching to a low-intensity task of organizing physics materials to reset focus and minimize digital distractions as per the reset_plan rule.,continue,anticipating viewer reactions to her speedrun physics segment,Lunch at her favorite cafe while catching up on messages.,Maria organizes her notes in the home office while her mind drifts to anticipating viewer reactions to her speedrun physics segment.,lunch,lunch,Hobbs_Cafe:main_floor,Hobbs_Cafe:main_floor


**Observations:**

ORPA:
- Drift layer `should_drift` value determines Act layer `drift_type`, `drift_topic`, `staet_summary_a` values.
  - This may be incorrect mechanism. Reflect layer's meta_rule is PFC. So, inhibitor strength makes more sense to come from Reflect alyer.
  - Need to create a testing methode to evaluate each layer's intended function vs. actual.
  - Need to check other literatures to see how they validated the function of each layer. 
- gemma3:27b-cloud (temp 1.0) -- is `state_summary_r` at t reflecting `state_summary_a` at t-1 correctly?
  - not as the exact text, but as conceptual alignmnet as an abstract reflection of past episodic memory

ORPDA:
- gemini-3-flash-preview:cloud (temp 0.0)(Hailey) --  
- gpt-oss:20b-cloud (temp 0.0) -- Plan and Action matches
- cogito-2.1:671b-cloud (temp 1.0) -- Matches location between Plan vs Action. However, `state_summery_a` of Action layer sometimes says different location. Morning routines are in bathroom (for both Hailey and Maria). But, often, it says they are doing something in the bedroom or at a cafe. 
  - This is observed in other models as well. Need to check if there's any correleation with temperature or specific model architecture.
- cogito-2.1:671b-cloud (temp 0.0) -- `state_summary_p` intended + `topic_a` + `drift_action_d` + `drfit_topic_a` => `state_summary_a`. This pattern is observed.
  - Need to check if it's common for all models and temperatures.
  - gemini-3-flash-preview:cloud (temp 1.0) -- `state_summary_p` intended + `topic_a` + `drift_action_d` + `drfit_topic_a` => `state_summary_a`. This pattern is observed.

Common:
- Observation layer has t-1 value as it's retrieving what happened in the past up to t-1 timestamp.
- Failed runs:
  - gemini-3-flash-preview:cloud (temp 0.0, 0.3, 0.8, 1.0)(Hailey) -- Failed-- ORPDA x5 times, ORPA x1 time.
  - cogito-2.1:671b-cloud (temp 0.0)(Maria) -- Failed -- ORPDA x1 time
  - ran and failed again with (Hailey), closed all other applications on local computer. power is plugged. Monitored 12GB of 16GB RAM used throughout. 44GB disk space available.
  - tried and failed again (Maria WITH temp 0.0 ORPDA) at 8th tick. 
  - Tried in Github codespace. It's running OK at 31st tick currently (ORPA temp 1.0)(Hailey).

